# Layer 2: audio-plus-context models

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [ ]:
PROJECT_ROOT = next((p for p in [Path("/workspace"), Path.cwd(), Path.cwd().parent]
                     if (p / "output" / "10_layer2_features").exists()), Path.cwd())
DATASET_DIR = PROJECT_ROOT / "output" / "09_layer2_dataset"
FEATURE_DIR = PROJECT_ROOT / "output" / "10_layer2_features"
LABEL_DIR = PROJECT_ROOT / "output" / "11_layer2_proxy_labels"
AUDIO_ONLY_DIR = PROJECT_ROOT / "output" / "12_layer2_models_audio_only"
OUT_DIR = PROJECT_ROOT / "output" / "13_layer2_models_audio_context"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = LOG_DIR / f"12_layer2_audio_context_models_{RUN_TAG}.log"

def log(msg: str):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

log("Notebook started.")

In [ ]:
def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)

def load_clip_index(filename: str = "clip_feature_index.csv.gz") -> pd.DataFrame:
    path = FEATURE_DIR / filename
    df = read_csv_auto(path)
    if "_record_key" not in df.columns:
        raise ValueError(f"_record_key not found in clip index: {path}")
    return df

def load_feature_table_csv(filename: str, clip_index_df: pd.DataFrame | None = None) -> pd.DataFrame:
    path = FEATURE_DIR / filename
    df = read_csv_auto(path)

    if "_record_key" in df.columns:
        return df

    if clip_index_df is None:
        raise ValueError(f"_record_key not found in {path}, and no clip index was provided.")

    if len(df) != len(clip_index_df):
        raise ValueError(
            f"Row mismatch while attaching _record_key to {path}: "
            f"feature_rows={len(df)} vs clip_index_rows={len(clip_index_df)}"
        )

    df = df.reset_index(drop=True)
    keys = clip_index_df[["_record_key"]].reset_index(drop=True)
    return pd.concat([keys, df], axis=1)

def load_sparse_npz(filename: str) -> sparse.csr_matrix:
    path = FEATURE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return load_npz(path).tocsr()

def safe_json_dump(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def normalize_binary_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(int)
    if str(s.dtype).startswith("int") or str(s.dtype).startswith("float"):
        return s.fillna(0).astype(int)
    mapping = {
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "y": 1, "n": 0,
        "1": 1, "0": 0,
    }
    return s.astype(str).str.strip().str.lower().map(mapping).fillna(0).astype(int)

In [ ]:
label_candidates = [
    LABEL_DIR / "layer2_proxy_labels_master.csv.gz",
    LABEL_DIR / "layer2_proxy_labels_master.csv",
    LABEL_DIR / "proxy_labels_master.csv.gz",
    LABEL_DIR / "proxy_labels_master.csv",
]
label_path = next((p for p in label_candidates if p.exists()), None)
if label_path is None:
    raise FileNotFoundError(f"Could not find proxy label master file in {LABEL_DIR}")

labels_df = read_csv_auto(label_path)
log(f"Loaded labels: {label_path.name} shape={labels_df.shape}")
display(labels_df.head(3))

In [ ]:
master_candidates = [
    DATASET_DIR / "layer2_master_table.csv.gz",
    DATASET_DIR / "layer2_master_table.csv",
]
master_path = next((p for p in master_candidates if p.exists()), None)
if master_path is None:
    raise FileNotFoundError(f"Could not find layer2 master table in {DATASET_DIR}")

master_df = read_csv_auto(master_path)
if "_record_key" not in master_df.columns:
    raise ValueError("Layer2 master table must contain _record_key")

log(f"Loaded master table: {master_path.name} shape={master_df.shape}")

In [ ]:
clip_index = load_clip_index("clip_feature_index.csv.gz")
log(f"Loaded clip_feature_index: {clip_index.shape}")

dense_feature_files = {
    "sequence_summary_features": "sequence_summary_features.csv.gz",
    "unigram_props_full": "unigram_props_full.csv.gz",
    "duration_weighted_prop_full": "duration_weighted_prop_full.csv.gz",
    "unigram_props_high_bias_filtered": "unigram_props_high_bias_filtered.csv.gz",
    "duration_weighted_prop_high_bias_filtered": "duration_weighted_prop_high_bias_filtered.csv.gz",
}

dense_features = {}
for name, fname in dense_feature_files.items():
    df = load_feature_table_csv(fname, clip_index_df=clip_index)
    feat_cols = [c for c in df.columns if c != "_record_key"]
    dense_features[name] = {"df": df, "feature_cols": feat_cols}
    log(f"Loaded dense feature set {name}: rows={len(df)}, cols={len(feat_cols)}")

sparse_feature_files = {
    "tfidf_1to3_full": "tfidf_1to3_full.npz",
    "transition_probs_full": "transition_probs_full.npz",
    "tfidf_1to3_high_bias_filtered": "tfidf_1to3_high_bias_filtered.npz",
    "transition_probs_high_bias_filtered": "transition_probs_high_bias_filtered.npz",
}
sparse_features = {}
for name, fname in sparse_feature_files.items():
    mat = load_sparse_npz(fname)
    sparse_features[name] = mat
    log(f"Loaded sparse feature set {name}: shape={mat.shape}")

In [ ]:
model_base = clip_index[["_record_key"]].drop_duplicates().copy()
model_base = model_base.merge(labels_df, on="_record_key", how="left", validate="one_to_one")
model_base = model_base.merge(master_df, on="_record_key", how="left", suffixes=("", "__master"))

required_label_cols = [
    "proxy_functional_v1",
    "is_nonvocal",
    "is_distress_related",
    "is_resource_related",
]
missing_required = [c for c in required_label_cols if c not in model_base.columns]
if missing_required:
    raise ValueError(f"Missing required label columns: {missing_required}")

for col in ["is_nonvocal", "is_distress_related", "is_resource_related"]:
    model_base[col] = normalize_binary_series(model_base[col])

log(f"Model base shape={model_base.shape}")
display(model_base.head(2))

In [ ]:
candidate_context_cols = [
    "farm",
    "barn_zone",
    "barnzone",
    "zone",
    "microphone",
    "mic",
    "mic_placement_context",
    "placement_context",
    "day",
    "date",
    "time",
    "time_of_day",
]

selected_context_cols = [c for c in candidate_context_cols if c in model_base.columns]
if not selected_context_cols:
    raise ValueError(
        "No context columns found in layer2 master table. "
        "Expected one or more of farm, zone, microphone, placement, date/day/time."
    )

context_df = model_base[["_record_key"] + selected_context_cols].copy()

for col in selected_context_cols:
    if pd.api.types.is_datetime64_any_dtype(context_df[col]):
        context_df[col] = context_df[col].astype(str)
    elif pd.api.types.is_numeric_dtype(context_df[col]):
        context_df[col] = context_df[col].astype(str)
    else:
        context_df[col] = context_df[col].astype(str).fillna("MISSING").replace({"nan": "MISSING", "None": "MISSING", "": "MISSING"})

if "time" in context_df.columns and "time_of_day" not in context_df.columns:
    def _time_bin(v: str) -> str:
        try:
            hh = int(str(v).split(":")[0])
            if 5 <= hh < 12:
                return "morning"
            if 12 <= hh < 17:
                return "afternoon"
            if 17 <= hh < 21:
                return "evening"
            return "night"
        except Exception:
            return "unknown"
    context_df["time_bin"] = context_df["time"].apply(_time_bin)
    selected_context_cols = selected_context_cols + ["time_bin"]

display(context_df.head(3))
log(f"Selected context columns: {selected_context_cols}")

In [ ]:
context_only_df = context_df[selected_context_cols].copy()
context_only_df = context_only_df.fillna("MISSING").astype(str)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_context = encoder.fit_transform(context_only_df)
context_feature_names = encoder.get_feature_names_out(selected_context_cols).tolist()

log(f"Context matrix shape={X_context.shape}")
display(pd.DataFrame({"context_feature_name": context_feature_names}).head(20))

In [ ]:
dense_X = {}
for name, payload in dense_features.items():
    df = payload["df"]
    feat_cols = payload["feature_cols"]
    merged = model_base[["_record_key"]].merge(df, on="_record_key", how="left", validate="one_to_one")
    X = merged[feat_cols].fillna(0.0).to_numpy(dtype=np.float32)
    dense_X[name] = {"X": X, "feature_cols": feat_cols}
    log(f"Aligned dense {name}: shape={X.shape}")

n_rows_expected = len(model_base)
for name, mat in sparse_features.items():
    if mat.shape[0] != n_rows_expected:
        raise ValueError(f"Sparse matrix row mismatch for {name}: {mat.shape[0]} vs {n_rows_expected}")

In [ ]:
audio_feature_registry = [
    {"feature_name": "sequence_summary_features", "kind": "dense"},
    {"feature_name": "unigram_props_full", "kind": "dense"},
    {"feature_name": "duration_weighted_prop_full", "kind": "dense"},
    {"feature_name": "tfidf_1to3_full", "kind": "sparse"},
    {"feature_name": "transition_probs_full", "kind": "sparse"},
    {"feature_name": "unigram_props_high_bias_filtered", "kind": "dense"},
    {"feature_name": "duration_weighted_prop_high_bias_filtered", "kind": "dense"},
    {"feature_name": "tfidf_1to3_high_bias_filtered", "kind": "sparse"},
    {"feature_name": "transition_probs_high_bias_filtered", "kind": "sparse"},
]

feature_registry_rows = []
for row in audio_feature_registry:
    feature_registry_rows.append({
        "feature_bundle": f"{row['feature_name']}__plus_context",
        "audio_feature_name": row["feature_name"],
        "audio_kind": row["kind"],
        "context_added": True,
    })
feature_registry_df = pd.DataFrame(feature_registry_rows)
display(feature_registry_df.head(10))

In [ ]:
target_specs = [
    {"target_name": "proxy_functional_v1", "task_type": "multiclass", "min_class_count_required": 5},
    {"target_name": "is_nonvocal", "task_type": "binary", "min_class_count_required": 20},
    {"target_name": "is_distress_related", "task_type": "binary", "min_class_count_required": 20},
    {"target_name": "is_resource_related", "task_type": "binary", "min_class_count_required": 20},
]

target_summary_rows = []
for spec in target_specs:
    y = model_base[spec["target_name"]]
    counts = y.value_counts(dropna=False).to_dict()
    target_summary_rows.append({
        "target_name": spec["target_name"],
        "task_type": spec["task_type"],
        "class_counts": counts,
        "n_classes": int(y.nunique()),
        "min_class_count": int(y.value_counts().min()),
        "eligible": int(y.value_counts().min()) >= spec["min_class_count_required"],
    })
target_summary_df = pd.DataFrame(target_summary_rows)
display(target_summary_df)

In [ ]:
def choose_n_splits(y: pd.Series, preferred: int = 5, min_required: int = 2) -> int:
    class_counts = y.value_counts()
    min_count = int(class_counts.min())
    n_splits = min(preferred, min_count)
    if n_splits < min_required:
        raise ValueError(f"Not enough samples per class for CV: min_count={min_count}")
    return n_splits

def make_model(task_type: str, feature_kind: str):
    if task_type == "multiclass":
        return LogisticRegression(
            max_iter=4000,
            solver="saga",
            class_weight="balanced",
            random_state=42,
        )
    return LogisticRegression(
        max_iter=4000,
        solver="liblinear",
        class_weight="balanced",
        random_state=42,
    )

def make_dummy():
    return DummyClassifier(strategy="most_frequent")

def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X), "proba"
    if hasattr(model, "decision_function"):
        return model.decision_function(X), "decision"
    return None, None

def multiclass_metrics(y_true, y_pred, y_proba=None, labels=None):
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    if y_proba is not None:
        try:
            out["log_loss"] = float(log_loss(y_true, y_proba, labels=labels))
        except Exception:
            out["log_loss"] = None
    return out

def binary_metrics(y_true, y_pred, y_score=None, y_proba=None):
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }
    if y_proba is not None:
        try:
            out["log_loss"] = float(log_loss(y_true, np.vstack([1-y_proba, y_proba]).T))
        except Exception:
            out["log_loss"] = None
    if y_score is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_score))
        except Exception:
            out["roc_auc"] = None
    return out

In [ ]:
feature_bundles = {}
for row in audio_feature_registry:
    feature_name = row["feature_name"]
    kind = row["kind"]

    if kind == "dense":
        X_audio = csr_matrix(dense_X[feature_name]["X"])
    else:
        X_audio = sparse_features[feature_name].tocsr()

    X_bundle = hstack([X_audio, X_context], format="csr")
    bundle_name = f"{feature_name}__plus_context"
    feature_bundles[bundle_name] = {
        "X": X_bundle,
        "audio_feature_name": feature_name,
        "audio_kind": kind,
        "context_dim": X_context.shape[1],
        "total_dim": X_bundle.shape[1],
    }
    log(f"Built bundle {bundle_name}: shape={X_bundle.shape}")

In [ ]:
audio_only_diag_path = AUDIO_ONLY_DIR / "audio_only_diagnostics.json"
audio_only_diag = None
if audio_only_diag_path.exists():
    with open(audio_only_diag_path, "r", encoding="utf-8") as f:
        audio_only_diag = json.load(f)
    log("Loaded audio-only diagnostics for later comparison.")

In [ ]:
all_metrics = []
all_fold_preds = []
confusion_store = []

for spec in target_specs:
    target_name = spec["target_name"]
    task_type = spec["task_type"]
    y_raw = model_base[target_name].copy()

    class_counts = y_raw.value_counts()
    min_count = int(class_counts.min())
    if min_count < spec["min_class_count_required"]:
        log(f"Skipping target={target_name}: min_class_count={min_count} < required={spec['min_class_count_required']}")
        continue

    n_splits = choose_n_splits(y_raw, preferred=5, min_required=2)
    session_group = model_base["_record_key"].map(clip_index.set_index("_record_key")[["farm", "barn_zone", "microphone", "mic_placement_context"]].astype(str).agg("|".join, axis=1)).to_numpy()
    cv = StratifiedGroupKFold(n_splits=n_splits)
    log(f"Evaluating target={target_name}, task_type={task_type}, n_splits={n_splits}, class_counts={class_counts.to_dict()}")

    if task_type == "multiclass":
        le = LabelEncoder()
        y = le.fit_transform(y_raw.astype(str))
        label_names = le.classes_.tolist()
    else:
        y = y_raw.astype(int).to_numpy()
        label_names = [0, 1]

    for bundle_name, payload in feature_bundles.items():
        X = payload["X"]

        for model_name, estimator in [("dummy_most_frequent", make_dummy()), ("logreg_balanced", make_model(task_type, "sparse"))]:
            y_true_all = []
            y_pred_all = []
            y_score_all = []
            y_proba_all = []

            for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(np.zeros(len(y)), y, session_group), start=1):
                X_tr = X[tr_idx]
                X_te = X[te_idx]
                y_tr = y[tr_idx]
                y_te = y[te_idx]

                est = clone(estimator)
                est.fit(X_tr, y_tr)
                y_pred = est.predict(X_te)

                score_obj, score_kind = predict_scores(est, X_te)

                fold_pred_df = pd.DataFrame({
                    "_record_key": model_base.iloc[te_idx]["_record_key"].values,
                    "target_name": target_name,
                    "feature_bundle": bundle_name,
                    "audio_feature_name": payload["audio_feature_name"],
                    "context_added": True,
                    "model_name": model_name,
                    "fold": fold_idx,
                    "y_true": y_te,
                    "y_pred": y_pred,
                })

                if score_kind == "proba":
                    if task_type == "multiclass":
                        y_proba_all.extend(score_obj.tolist())
                        for j, cls_name in enumerate(label_names):
                            fold_pred_df[f"proba__{cls_name}"] = score_obj[:, j]
                    else:
                        y_proba_all.extend(score_obj[:, 1].tolist())
                        y_score_all.extend(score_obj[:, 1].tolist())
                        fold_pred_df["proba__1"] = score_obj[:, 1]
                elif score_kind == "decision" and task_type == "binary":
                    scores_1d = np.asarray(score_obj).reshape(-1)
                    y_score_all.extend(scores_1d.tolist())
                    fold_pred_df["score__decision"] = scores_1d

                all_fold_preds.append(fold_pred_df)
                y_true_all.extend(y_te.tolist())
                y_pred_all.extend(y_pred.tolist())

            y_true_all = np.asarray(y_true_all)
            y_pred_all = np.asarray(y_pred_all)

            if task_type == "multiclass":
                y_proba_mat = np.asarray(y_proba_all) if len(y_proba_all) else None
                metrics = multiclass_metrics(
                    y_true=y_true_all,
                    y_pred=y_pred_all,
                    y_proba=y_proba_mat if len(y_proba_all) else None,
                    labels=np.arange(len(label_names)),
                )
                cm = confusion_matrix(y_true_all, y_pred_all, labels=np.arange(len(label_names)))
                cm_df = pd.DataFrame(cm, index=[f"true__{x}" for x in label_names], columns=[f"pred__{x}" for x in label_names])
            else:
                y_score_vec = np.asarray(y_score_all) if len(y_score_all) else None
                y_proba_vec = np.asarray(y_proba_all) if len(y_proba_all) else None
                metrics = binary_metrics(
                    y_true=y_true_all,
                    y_pred=y_pred_all,
                    y_score=y_score_vec if len(y_score_all) else None,
                    y_proba=y_proba_vec if len(y_proba_all) else None,
                )
                cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
                cm_df = pd.DataFrame(cm, index=["true__0", "true__1"], columns=["pred__0", "pred__1"])

            all_metrics.append({
                "run_ts_utc": RUN_TS,
                "target_name": target_name,
                "task_type": task_type,
                "feature_bundle": bundle_name,
                "audio_feature_name": payload["audio_feature_name"],
                "context_added": True,
                "context_dim": int(payload["context_dim"]),
                "total_dim": int(payload["total_dim"]),
                "model_name": model_name,
                "n_splits": n_splits,
                "n_samples": len(y_true_all),
                "n_classes": len(np.unique(y_true_all)),
                **metrics,
            })

            cm_key = f"{target_name}__{bundle_name}__{model_name}"
            confusion_store.append((cm_key, cm_df))
            log(f"Done target={target_name} bundle={bundle_name} model={model_name} metrics={metrics}")

In [ ]:
metrics_df = pd.DataFrame(all_metrics).sort_values(
    by=["target_name", "model_name", "macro_f1", "balanced_accuracy"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

fold_preds_df = pd.concat(all_fold_preds, ignore_index=True) if all_fold_preds else pd.DataFrame()

metrics_path = OUT_DIR / "audio_context_metrics_summary.csv"
fold_preds_path = OUT_DIR / "audio_context_fold_predictions.csv.gz"
manifest_path = OUT_DIR / "audio_context_manifest.json"
context_vocab_path = OUT_DIR / "context_feature_names.csv"

metrics_df.to_csv(metrics_path, index=False)
if not fold_preds_df.empty:
    fold_preds_df.to_csv(fold_preds_path, index=False, compression="gzip")

pd.DataFrame({"context_feature_name": context_feature_names}).to_csv(context_vocab_path, index=False)

cm_dir = OUT_DIR / "confusion_matrices"
cm_dir.mkdir(parents=True, exist_ok=True)
for key, cm_df in confusion_store:
    cm_df.to_csv(cm_dir / f"{key}.csv")

comparison_df = None
if audio_only_diag is not None:
    rows = []
    for target_name, grp in metrics_df.groupby("target_name"):
        best_ctx = grp[grp["model_name"] == "logreg_balanced"].sort_values(["macro_f1", "balanced_accuracy"], ascending=[False, False]).iloc[0]
        audio_only_best = audio_only_diag.get("best_macro_f1_by_target", {}).get(target_name, {})
        rows.append({
            "target_name": target_name,
            "best_audio_only_feature": audio_only_best.get("feature_name"),
            "best_audio_only_macro_f1": audio_only_best.get("macro_f1"),
            "best_audio_only_balanced_accuracy": audio_only_best.get("balanced_accuracy"),
            "best_audio_context_bundle": best_ctx["feature_bundle"],
            "best_audio_context_macro_f1": best_ctx["macro_f1"],
            "best_audio_context_balanced_accuracy": best_ctx["balanced_accuracy"],
            "delta_macro_f1": None if audio_only_best.get("macro_f1") is None else float(best_ctx["macro_f1"] - audio_only_best["macro_f1"]),
            "delta_balanced_accuracy": None if audio_only_best.get("balanced_accuracy") is None else float(best_ctx["balanced_accuracy"] - audio_only_best["balanced_accuracy"]),
        })
    comparison_df = pd.DataFrame(rows)
    comparison_df.to_csv(OUT_DIR / "audio_vs_audio_context_comparison.csv", index=False)

manifest = {
    "build_ts_utc": RUN_TS,
    "dataset_dir": str(DATASET_DIR),
    "feature_dir": str(FEATURE_DIR),
    "label_dir": str(LABEL_DIR),
    "output_dir": str(OUT_DIR),
    "n_clips": int(len(model_base)),
    "label_source_file": str(label_path),
    "master_source_file": str(master_path),
    "selected_context_cols": selected_context_cols,
    "context_matrix_shape": [int(X_context.shape[0]), int(X_context.shape[1])],
    "primary_target": "proxy_functional_v1",
    "targets_modeled": sorted(metrics_df["target_name"].unique().tolist()) if not metrics_df.empty else [],
    "audio_feature_sets_used": sorted(metrics_df["audio_feature_name"].unique().tolist()) if not metrics_df.empty else [],
    "models": sorted(metrics_df["model_name"].unique().tolist()) if not metrics_df.empty else [],
    "outputs": {
        "metrics_summary_csv": metrics_path.name,
        "fold_predictions_csv_gz": fold_preds_path.name,
        "context_feature_names_csv": context_vocab_path.name,
        "confusion_matrices_dir": cm_dir.name,
        "audio_vs_audio_context_comparison_csv": "audio_vs_audio_context_comparison.csv" if comparison_df is not None else None,
    }
}
safe_json_dump(manifest, manifest_path)

diag = {
    "n_clips": int(len(model_base)),
    "selected_context_cols": selected_context_cols,
    "context_matrix_shape": [int(X_context.shape[0]), int(X_context.shape[1])],
    "targets_modeled": sorted(metrics_df["target_name"].unique().tolist()) if not metrics_df.empty else [],
    "best_macro_f1_by_target": {},
}
for target_name, grp in metrics_df[metrics_df["model_name"] == "logreg_balanced"].groupby("target_name"):
    best_row = grp.sort_values(["macro_f1", "balanced_accuracy"], ascending=[False, False]).iloc[0]
    diag["best_macro_f1_by_target"][target_name] = {
        "feature_bundle": best_row["feature_bundle"],
        "audio_feature_name": best_row["audio_feature_name"],
        "macro_f1": float(best_row["macro_f1"]),
        "balanced_accuracy": float(best_row["balanced_accuracy"]),
    }

safe_json_dump(diag, OUT_DIR / "audio_context_diagnostics.json")

log(f"Saved metrics: {metrics_path}")
log(f"Saved manifest: {manifest_path}")
if comparison_df is not None:
    log("Saved audio vs audio+context comparison.")
log("Notebook completed.")

In [ ]:
display(metrics_df.head(20))
display(target_summary_df)

diag_path = OUT_DIR / "audio_context_diagnostics.json"
with open(diag_path, "r", encoding="utf-8") as f:
    diag = json.load(f)
diag